In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *


In [0]:
df_silver=spark.sql("select *from parquet.`abfss://silver@carstorage1234.dfs.core.windows.net/rawdata`")

In [0]:
df_branch=spark.sql("select *from prod_car.gold.dim_branch")
df_model=spark.sql("select *from prod_car.gold.dim_model")
df_dealer=spark.sql("select *from prod_car.gold.dim_dealer")
df_date=spark.sql("select *from prod_car.gold.dim_date")

In [0]:
df_fact=df_silver.join(df_branch,df_silver.Branch_ID==df_branch.branch_id,"left").join(df_model,df_silver.Model_ID==df_model.model_id,"left").join(df_dealer,df_silver.Dealer_ID==df_dealer.dealer_id,"left").join(df_date,df_silver.Date_ID==df_date.date_id,"left").select(df_silver['Revenue'],df_silver['Units_Sold'],df_silver['Revperunit'],df_branch['dim_branch_key'],df_model['dim_model_key'],df_dealer['dim_dealer_key'],df_date['dim_date_key'])


In [0]:
df_fact.display()

Revenue,Units_Sold,Revperunit,dim_branch_key,dim_model_key,dim_dealer_key,dim_date_key
13363978,2,6681989.0,814,195,51,1001
17376468,3,5792156.0,815,261,131,1001
9664767,3,3221589.0,1,148,67,867
5525304,3,1841768.0,357,87,52,867
12971088,3,4323696.0,683,20,132,69
7321228,1,7321228.0,358,242,36,285
11379294,2,5689647.0,1492,160,82,285
11611234,2,5805617.0,933,179,119,1069
19979446,2,9989723.0,1493,262,239,1069
14181510,3,4727170.0,1493,209,1,1002


In [0]:
if spark.catalog.tableExists("Factsales"):
    deltatble=DeltaTable.forName(spark,"prod_car.gold.Factsales")
    deltatble.alias("t").merge(df_fact.alias("s"),"t.dim_branch_key=s.dim_branch_key and t.dim_model_key=s.dim_model_key and t.dim_dealer_key=s.dim_dealer_key and t.dim_date_key=s.dim_date_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_fact.write.format("delta").mode("Overwrite").option("path","abfss://gold@carstorage1234.dfs.core.windows.net/Factsales").saveAsTable("prod_car.gold.Factsales")